# TTTN · Kaggle Evaluation + Automatic Hybrid (U-Net + SegFormer + VMamba)

Notebook này **không train lại**. Thí nghiệm chính luôn là: **Validation chọn policy → Test báo cáo**. Tách riêng phía sau là **Data audit toàn bộ Train/Val/Test** dùng policy đã chốt để tìm ứng viên label/mask sai hoặc điểm yếu từng model; audit này không chỉnh ngưỡng và không được đưa vào metric Test.

Trước khi Run All: chọn GPU **Tesla T4**, bật Internet, rồi Add Input gói `TTTN_Kaggle_Selected_Thesis_Experiments_Final_20260818.zip`. Gói này đã có cả wheel Mamba. Không dùng P100 vì wheel đã build cho T4/sm75.

In [ ]:
# ====== CẤU HÌNH DUY NHẤT CẦN CHỈNH ======
RUN_NAME = 'evaluation_seed42'

VAL_TILE_BATCH_SIZE = 16

# Logic cuối là PASS / DEFECT hoàn toàn tự động, không có REVIEW.
MAX_FNR = 0.02             # tối đa 2% Defect bị PASS trên Validation
MAX_AUTO_DEFECT_FPR = 0.10 # mục tiêu cho ngưỡng triage tự động
FNR_SAFETY_MARGIN = 0.005

# Data audit độc lập: chạy inference cả Train/Val/Test bằng policy đã chốt ở Val.
# Tốn thêm thời gian vì cần xuất map Train của cả ba model, nhưng KHÔNG train và KHÔNG đổi metric Test.
RUN_FULL_DATA_AUDIT = True
AUDIT_PREVIEW_LIMIT = 0  # 0 = tạo ảnh cho mọi case nghi vấn; đặt 500 nếu muốn chạy/gói nhẹ hơn


In [ ]:
# Cài đúng runtime tương thích với VMamba wheel. Chạy cell này trước khi import torch.
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'torch==2.11.0', 'torchvision==0.26.0', 'torchaudio==2.11.0',
    '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True)
import torch
assert torch.cuda.is_available(), 'Bật GPU trong Kaggle Settings'
print('Python:', sys.version.split()[0])
print('Torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0), '| CC:', torch.cuda.get_device_capability(0))
assert torch.cuda.get_device_capability(0) == (7, 5), 'Chọn Tesla T4, không dùng P100'


In [ ]:
# Tìm project-data input, copy code vào working và giữ dataset ở input read-only.
from pathlib import Path
import os, shutil, zipfile

INPUT = Path('/kaggle/input')
WORKING = Path('/kaggle/working')
markers = list(INPUT.rglob('run_all_decision_experiments.py'))
if markers:
    PROJECT_INPUT = markers[0].parents[2]
else:
    archive_patterns = ['TTTN_Kaggle_Selected_Thesis_Experiments_Final_*.zip', 'TTTN_Kaggle_All3_Evaluation_Complete_*.zip']
    archives = [path for pattern in archive_patterns for path in INPUT.rglob(pattern)]
    assert archives, 'Chưa Add Input gói TTTN Kaggle evaluation final'
    extracted = WORKING / 'tttn_all3_input'
    with zipfile.ZipFile(archives[0]) as zf:
        zf.extractall(extracted)
    markers = list(extracted.rglob('run_all_decision_experiments.py'))
    assert markers, 'Archive project không hợp lệ'
    PROJECT_INPUT = markers[0].parents[2]

PROJECT = WORKING / 'threecad_ani_project'
shutil.copytree(PROJECT_INPUT, PROJECT, dirs_exist_ok=True,
                ignore=shutil.ignore_patterns('data', 'artifacts', 'archive', '.venv', 'node_modules', '__pycache__', '.git'))
DATASET_ROOT = PROJECT_INPUT / 'data' / '3cad_ani'
assert (PROJECT / 'scripts/training/train_on_kaggle.py').is_file()
assert (PROJECT / 'scripts/experiments/run_all_decision_experiments.py').is_file()
assert (DATASET_ROOT / 'dataset_audit/splits/train.csv').is_file()

os.chdir(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements/ml-kaggle.txt'], check=True)
# Wheel nằm ngay trong gói evaluation; vẫn tìm toàn bộ Input để tương thích gói cũ.
# Khi input là ZIP, weights/wheel nằm trong thư mục đã extract ở WORKING; khi input đã mở rộng, chúng nằm ở /kaggle/input.
BUNDLE_SEARCH_ROOTS = [PROJECT_INPUT, PROJECT_INPUT.parent, PROJECT_INPUT.parent.parent, INPUT]
wheel_pattern = '*mamba_ssm-2.3.2.post1*cu128torch2.11sm75*cp312*linux_x86_64.whl'
wheel_candidates = [path for root in BUNDLE_SEARCH_ROOTS for path in root.rglob(wheel_pattern)]
wheel_candidates = list(dict.fromkeys(wheel_candidates))
assert wheel_candidates, 'Không thấy mamba_ssm wheel trong Kaggle Input'
wheel = wheel_candidates[0]  # Kaggle có thể bỏ dấu + trong tên file, nên không hard-code basename.
os.environ['PROJECT_ROOT'] = str(PROJECT)
os.environ['MAMBA_WHEEL'] = str(wheel)
subprocess.run(['bash', 'scripts/setup/setup_vmamba_kaggle.sh'], cwd=PROJECT, check=True)

RESULTS = WORKING / 'results'
PREDICTIONS = WORKING / 'probabilities'
DECISION_REPORT = WORKING / 'three_model_decision_report'
print('PROJECT:', PROJECT)
print('DATASET_ROOT:', DATASET_ROOT)


In [ ]:
# Kiểm tra split trước khi evaluation (không train).
subprocess.run([sys.executable, 'scripts/verification/check_protocol.py',
    '--dataset-root', str(DATASET_ROOT),
    '--train-csv', str(DATASET_ROOT / 'dataset_audit/splits/train.csv'),
    '--val-csv', str(DATASET_ROOT / 'dataset_audit/splits/val.csv'),
    '--test-csv', str(DATASET_ROOT / 'dataset_audit/splits/test.csv'),
    '--save', str(WORKING / 'protocol_check.json')], cwd=PROJECT, check=True)


In [ ]:
# Chỉ dùng checkpoint có sẵn: copy vào working để Validation/Test ghi metric và threshold bên cạnh weight.
folders = {'unet': 'unet_r18', 'segformer': 'segformer_b0', 'vmamba': 'vmamba_t_s2l5'}
weight_names = {'unet': 'unet_best.pt', 'segformer': 'segformer_best.pt', 'vmamba': 'vmamba_best.pt'}
CHECKPOINTS = {}
for model, folder in folders.items():
    weight_candidates = [path for root in BUNDLE_SEARCH_ROOTS for path in root.rglob(weight_names[model])]
    source = next(iter(weight_candidates), None)
    assert source is not None, f'Không thấy weight {weight_names[model]} trong Kaggle Input'
    target = RESULTS / folder / RUN_NAME / 'checkpoints/best.pt'
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, target)
    CHECKPOINTS[model] = target

# Validation chọn threshold của từng model; Test dùng đúng threshold đó, không dùng Test để chỉnh.
for model, checkpoint in CHECKPOINTS.items():
    common = [sys.executable, 'scripts/evaluation/evaluate_model.py', '--model', model,
        '--checkpoint', str(checkpoint), '--dataset-root', str(DATASET_ROOT),
        '--train-csv', str(DATASET_ROOT / 'dataset_audit/splits/train.csv'),
        '--val-csv', str(DATASET_ROOT / 'dataset_audit/splits/val.csv'),
        '--test-csv', str(DATASET_ROOT / 'dataset_audit/splits/test.csv'),
        '--tile-size', '512', '--stride', '256',
        '--tile-batch-size', str(VAL_TILE_BATCH_SIZE), '--warmup-batches', '2']
    print('\nVALIDATION:', model.upper())
    subprocess.run(common + ['--split', 'val'], cwd=PROJECT, check=True)
    print('\nTEST:', model.upper())
    subprocess.run(common + ['--split', 'test'], cwd=PROJECT, check=True)


In [ ]:
# Xuất full-resolution probability map. Val/Test dùng cho thí nghiệm; Train chỉ dùng cho data audit độc lập.
map_splits = ['train', 'val', 'test'] if RUN_FULL_DATA_AUDIT else ['val', 'test']
for model, checkpoint in CHECKPOINTS.items():
    output_root = PREDICTIONS / model
    command = [sys.executable, 'scripts/experiments/export_probability_cache.py',
        '--model', model, '--checkpoint', str(checkpoint),
        '--dataset-root', str(DATASET_ROOT), '--output-root', str(output_root),
        '--splits', *map_splits, '--tile-size', '512', '--stride', '256']
    print('\nEXPORT PROBABILITY:', model.upper())
    subprocess.run(command, cwd=PROJECT, check=True)

# Tất cả logic tự động: từng model riêng, fusion 3 model và hybrid từng cặp. Không xuất REVIEW.
decision_command = [sys.executable, 'scripts/experiments/run_all_decision_experiments.py',
    '--dataset-root', str(DATASET_ROOT), '--output-dir', str(DECISION_REPORT),
    '--max-fnr', str(MAX_FNR), '--max-defect-fpr', str(MAX_AUTO_DEFECT_FPR),
    '--fnr-safety-margin', str(FNR_SAFETY_MARGIN), '--folds', '5', '--feature-size', '256', '--automatic-only']
for model in ('unet', 'segformer', 'vmamba'):
    decision_command += ['--prediction', f'{model}={PREDICTIONS / model}']
subprocess.run(decision_command, cwd=PROJECT, check=True)

# DATA AUDIT RIÊNG: policy đã đóng băng từ Validation, không dùng để chọn lại threshold hay thay đổi bảng Test.
DATA_AUDIT = WORKING / 'full_dataset_data_audit'
if RUN_FULL_DATA_AUDIT:
    audit_command = [sys.executable, 'scripts/data/audit_full_dataset_labels.py',
        '--dataset-root', str(DATASET_ROOT),
        '--frozen-policy', str(DECISION_REPORT / 'adaptive_single/adaptive_component_policy.json'),
        '--output-dir', str(DATA_AUDIT), '--splits', 'train', 'val', 'test',
        '--preview-limit', str(AUDIT_PREVIEW_LIMIT), '--preview-size', '288']
    for model in ('unet', 'segformer', 'vmamba'):
        audit_command += ['--prediction', f'{model}={PREDICTIONS / model}']
    subprocess.run(audit_command, cwd=PROJECT, check=True)


In [ ]:
# Tổng hợp báo cáo đồ án đã chốt scope: E0, E2-E5, E7, E8 (không gồm E1/E6/E9/E10/E11).
import json, pandas as pd
base_rows = []
for model, checkpoint in CHECKPOINTS.items():
    metrics = json.loads((checkpoint.parent.parent / 'test/main_metrics.json').read_text())
    base_rows.append({
        'model': model, 'threshold': metrics['threshold'],
        'fnr_pct': 100 * metrics['image_fnr'], 'fpr_pct': 100 * metrics['image_fpr'],
        'positive_dice_pct': 100 * metrics['positive_dice'],
        'false_negatives': metrics['image_fn'], 'false_positives': metrics['image_fp']})
base = pd.DataFrame(base_rows)
base.to_csv(DECISION_REPORT / 'tables/00_base_model_segmentation_test.csv', index=False)
display(base)

THESIS_REPORT = WORKING / 'thesis_evaluation_report'
thesis_command = [sys.executable, 'scripts/reporting/compile_thesis_evaluation_report.py',
    '--unet-run', str(CHECKPOINTS['unet'].parent.parent),
    '--segformer-run', str(CHECKPOINTS['segformer'].parent.parent),
    '--vmamba-run', str(CHECKPOINTS['vmamba'].parent.parent),
    '--protocol-check', str(WORKING / 'protocol_check.json'),
    '--output-dir', str(THESIS_REPORT)]
subprocess.run(thesis_command, cwd=PROJECT, check=True)
print('Bao cao do an:', THESIS_REPORT)
display(pd.read_csv(THESIS_REPORT / 'tables/01_e2_architecture_comparison.csv'))

# Bảng logic/hybrid tự động và audit Test riêng.
master = pd.read_csv(DECISION_REPORT / 'tables/01_master_test_comparison.csv')
display(master)
automatic = pd.read_csv(DECISION_REPORT / 'tables/02_fully_automatic_comparison.csv')
display(automatic)

# Mỗi experiment được đối chiếu trên TOÀN BỘ Test: TP/TN/FN/FP.
# Tạo preview Input | GT | probability | PASS/DEFECT để kiểm tra trực quan.
audit_command = [sys.executable, 'scripts/reporting/build_test_case_audit.py',
    '--dataset-root', str(DATASET_ROOT), '--decision-report', str(DECISION_REPORT),
    '--preview-size', '288']
for model in ('unet', 'segformer', 'vmamba'):
    audit_command += ['--prediction', f'{model}={PREDICTIONS / model}']
    audit_command += ['--base-test', f'{model}={CHECKPOINTS[model].parent.parent / "test" / "per_image_metrics.csv"}']
subprocess.run(audit_command, cwd=PROJECT, check=True)
case_summary = pd.read_csv(DECISION_REPORT / 'tables/03_test_case_outcome_summary.csv')
display(case_summary)
print('Toàn bộ danh sách/case ảnh:', DECISION_REPORT / 'test_case_audit')
print('Preview all cases và lỗi:', DECISION_REPORT / 'test_case_audit/visual_gallery')
if RUN_FULL_DATA_AUDIT:
    data_audit_summary = pd.read_csv(DATA_AUDIT / '03_summary_by_split_and_reason.csv')
    display(data_audit_summary)
    print('Data audit tất cả ảnh:', DATA_AUDIT)
    print('Hàng đợi kiểm tra label/mask:', DATA_AUDIT / '02_review_queue.csv')
# Một ZIP gồm báo cáo đồ án, logic/hybrid và data audit (nếu bật).
DELIVERABLES = WORKING / 'final_thesis_deliverables'
if DELIVERABLES.exists():
    shutil.rmtree(DELIVERABLES)
shutil.copytree(THESIS_REPORT, DELIVERABLES / 'thesis_evaluation_report')
shutil.copytree(DECISION_REPORT, DELIVERABLES / 'decision_and_test_audit')
if RUN_FULL_DATA_AUDIT:
    shutil.copytree(DATA_AUDIT, DELIVERABLES / 'full_dataset_data_audit')
archive = shutil.make_archive(str(WORKING / 'final_thesis_deliverables'), 'zip', DELIVERABLES)
print('Tải toàn bộ báo cáo:', archive)
print('Lưu notebook bằng Save Version để Kaggle giữ results/checkpoints/probabilities.')
